### Import Required Libraries

In [12]:
import warnings
warnings.filterwarnings("ignore")

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv2D, MaxPooling2D, Flatten, Dense, 
                                     Dropout, BatchNormalization, Rescaling, 
                                     RandomFlip, RandomRotation, RandomZoom, 
                                     Input, GlobalAveragePooling2D)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

In [13]:
train_dir = 'preprocessed_images/train'
test_dir = 'preprocessed_images/validation'
model_save_path = 'cnn_custom_first_model.keras'

class_names = ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']
IMG_SIZE = (48, 48)
BATCH_SIZE = 128
EPOCHS = 100

In [14]:
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir,
    labels = 'inferred',
    label_mode = 'int',
    class_names = None,
    color_mode = 'rgb',
    batch_size = 32,
    image_size = (48, 48),
    shuffle = True,
)

test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    test_dir,
    labels = 'inferred',
    label_mode = 'int',
    class_names = None,
    color_mode = 'rgb',
    batch_size = 32,
    image_size = (48, 48),
    shuffle = True,
)

# Prefetch for performance
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.prefetch(tf.data.AUTOTUNE)



Found 27473 files belonging to 7 classes.
Found 6561 files belonging to 7 classes.


In [15]:
model = Sequential([
    Input(shape = (48,48,1)),
    
    # Data Augmentation
    
    Rescaling(1./255),
    RandomFlip("horizontal"),
    RandomRotation(0.1),
    RandomZoom(0.1),
    
    # Set of 1st Neural Network
    Conv2D(64, (3,3), padding = "same", activation = 'relu'),
    BatchNormalization(), 
    Conv2D(64, (3,3), padding = "same", activation = 'relu'),
    BatchNormalization(), 
    MaxPooling2D((2,2)),
    Dropout(0.25),
    
    # Set of 2nd Neural Network
    Conv2D(128, (3,3), padding = "same", activation = 'relu'),
    BatchNormalization(), 
    Conv2D(128, (3,3), padding = "same", activation = 'relu'),
    BatchNormalization(), 
    MaxPooling2D((2,2)),
    Dropout(0.25),
    
    # Set of 3rd Neural Network
    Conv2D(256, (3,3), padding = "same", activation = 'relu'),
    BatchNormalization(), 
    Conv2D(256, (3,3), padding = "same", activation = 'relu'),
    BatchNormalization(), 
    MaxPooling2D((2,2)),
    Dropout(0.25),
    
    # Dense Layer
    GlobalAveragePooling2D(),
    Dense(256, activation = 'relu'),
    Dropout(0.25),
    
    # Last set of Neural Network
    Dense(7, activation= 'softmax')
    
])
model.summary()





Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_1 (Rescaling)         │ (None, 48, 48, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_flip_1 (RandomFlip)      │ (None, 48, 48, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation_1               │ (None, 48, 48, 1)      │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_zoom_1 (RandomZoom)      │ (None, 48, 48, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 48, 48, 64)     │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 48, 48, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 48, 48, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 48, 48, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 24, 24, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 24, 24, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 24, 24, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 24, 24, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_9 (Conv2D)               │ (None, 24, 24, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 24, 24, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 12, 12, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 12, 12, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 12, 12, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_10          │ (None, 12, 12, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (None, 12, 12, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_11          │ (None, 12, 12, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 6, 6, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 1,215,431 (4.64 MB)

 Trainable params: 1,213,639 (4.63 MB)

 Non-trainable params: 1,792 (7.00 KB)

In [16]:
model.compile(
    optimizer = "adam",
    loss = "SparseCategoricalCrossentropy",
    metrics = ["SparseCategoricalAccuracy"]   
)

In [18]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        "first_model.keras",
        monitor = 'val_loss',
        verbose = 1,
        save_best_only=True,
        save_weights_only=False,
        mode = 'auto',
        save_freq = 'epoch',
        initial_value_threshold=None
    ),
    
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        min_delta=0.01,
        patience=5,
        verbose=1,
        mode='auto',
        baseline=None,
        restore_best_weights=True,
        start_from_epoch=0
    ),
    
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=10,
        verbose=1,
        mode='auto',
    )
    
]
model.fit(
    train_ds,
    validation_data = test_ds,
    epochs = 20,
    callbacks = callbacks
)


Epoch 1/20


ValueError: Exception encountered when calling Sequential.call().

[1mInput 0 with name 'None' of layer 'conv2d_6' is incompatible with the layer: expected axis -1 of input shape to have value 1, but received input with shape (None, 48, 48, 3)[0m

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=(None, 48, 48, 3), dtype=float32)
  • training=True
  • mask=None
  • kwargs=<class 'inspect._empty'>